# RIFE desde cero — run 2 en Kaggle (2×T4): RIFE-m fine-tune sobre Vimeo **septuplet**

Guía completa en `docs/KAGGLE.md` (sección *Run 2*).  Configura **Accelerator: GPU T4 x2** e **Internet: ON**.

**Qué cambia respecto al run 1** (60 ep triplet → 34.33 dB):

| | run 1 | run 2 |
|---|---|---|
| Dataset | `vimeo_triplet` (51k tríos, t=0.5) | `vimeo_septuplet` (64k secuencias de 7, t aleatorio) |
| Modelo | RIFE | **RIFE-m** (`--arbitrary_time`: canal t en los 4 IFBlocks) |
| Inicialización | aleatoria | **`--init_from` best.pth del run 1** (canal t a cero → arranca idéntico, fine-tune barato) |
| Distill weight | 0.01 | 0.005 (recomendación del paper para RIFE-m) |
| EMA | no | **`--ema 0.999`** (se valida y exporta con pesos EMA) |
| Scale aug | no | **`--scale_aug 0.5 0.5 1.5`** (consejo de hzwer para movimiento grande) |
| LR pico | 1.5e-4 | 1e-4, warmup 500 (fine-tune) |

**Inputs necesarios**: (1) el dataset septuplet — busca en *Add Input → Datasets* `vimeo septuplet` (p.ej. `maiimaii/vimeo-septuplet`, 88 GB, o `wangsally/vimeo-90k-7`); la celda de abajo lo localiza por `sep_trainlist.txt`.  (2) La versión guardada del notebook del run 1 (*Your Work → Notebooks*) para leer `checkpoints/best.pth`.

Para reanudar en la 2ª/3ª sesión: añade la versión anterior de ESTE notebook como input y pon `RESUME` (ver última celda).


In [ ]:
import glob, os

# --- Configuración ---
EPOCHS      = 90        # TOTAL del plan (≈45 ep/sesión de 12 h con septuplet); NO lo cambies entre sesiones
BATCH       = 16        # por GPU
REFINE_C    = 16        # DEBE coincidir con el run 1 (si cambias, el UNet arranca aleatorio y pierdes el fine-tune)
LR          = 1e-4      # pico (fine-tune); run 1 usó 1.5e-4 efectivos
WARMUP      = 500
EMA         = 0.999
SCALE_AUG   = "0.5 0.5 1.5"   # P SMIN SMAX
TIME_LIMIT  = 11.2      # horas; para limpiamente antes de las 12 h de Kaggle
INIT_FROM   = None      # 1ª sesión: best.pth del run 1, p.ej. glob.glob('/kaggle/input/*/checkpoints/best.pth')[0]
RESUME      = None      # 2ª+ sesión: last.pth de este run, p.ej. glob.glob('/kaggle/input/*/checkpoints_run2/last.pth')[0]
SMOKE_FIRST = False     # True → smoke test de 3 min antes del entrenamiento real
MULTI_T_EVAL = True     # evaluación ×6 (t=k/6) al final, además de la estándar t=0.5

# Localiza el dataset septuplet (la carpeta que contiene sep_trainlist.txt)
cands = glob.glob('/kaggle/input/**/sep_trainlist.txt', recursive=True)
assert cands, "No encuentro sep_trainlist.txt: añade un mirror de vimeo_septuplet como Input"
DATA_ROOT = os.path.dirname(cands[0])
OUT = '/kaggle/working/checkpoints_run2'
print('DATA_ROOT =', DATA_ROOT)
print('secuencias train:', sum(1 for _ in open(f'{DATA_ROOT}/sep_trainlist.txt')))

# Localiza automáticamente el checkpoint del run 1 si no se indicó
if INIT_FROM is None and RESUME is None:
    c = sorted(glob.glob('/kaggle/input/**/best.pth', recursive=True))
    assert c, "Añade la versión del notebook del run 1 como Input (contiene checkpoints/best.pth) o pon INIT_FROM"
    INIT_FROM = c[0]
print('INIT_FROM =', INIT_FROM, '| RESUME =', RESUME)


In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
!rm -rf /kaggle/working/rife && git clone -q https://github.com/correo415415/super-light-resolution.git /kaggle/working/rife
%cd /kaggle/working/rife
!pip install -q tensorboard 2>/dev/null
!python tests/test_core.py | tail -3


In [ ]:
if SMOKE_FIRST:
    !torchrun --nproc_per_node=2 train.py --data_root {DATA_ROOT} --smoke --amp --num_workers 2 \
        --arbitrary_time --init_from {INIT_FROM} --ema {EMA} --scale_aug {SCALE_AUG} --out_dir /kaggle/working/smoke


In [ ]:
init_flag = f'--init_from {INIT_FROM}' if (INIT_FROM and not RESUME) else ''
resume_flag = f'--resume {RESUME}' if RESUME else ''
!torchrun --nproc_per_node=2 train.py \
    --data_root {DATA_ROOT} --out_dir {OUT} {init_flag} {resume_flag} \
    --arbitrary_time --ema {EMA} --scale_aug {SCALE_AUG} \
    --epochs {EPOCHS} --batch_size {BATCH} --refine_c {REFINE_C} \
    --lr {LR} --lr_min 2e-6 --warmup_steps {WARMUP} --no_lr_scale --amp --num_workers 4 \
    --time_limit {TIME_LIMIT} --save_every 500 --log_every 100


In [ ]:
# Evaluación del mejor checkpoint (pesos EMA se cargan solos).  t=0.5 sobre el test septuplet (im3→im5, GT im4)
best = f'{OUT}/best.pth' if os.path.exists(f'{OUT}/best.pth') else f'{OUT}/last.pth'
!python evaluate.py --ckpt {best} --data_root {DATA_ROOT} --amp --max_samples 1000 \
    --vis_dir /kaggle/working/vis --n_vis 4 --out_json /kaggle/working/eval_t05.json
if MULTI_T_EVAL:
    # ×6: im1→im7 con t=1/6..5/6.  Comprueba que el canal t se usa (los extremos deben ir ≈ igual que el centro).
    !python evaluate.py --ckpt {best} --data_root {DATA_ROOT} --amp --max_samples 1000 --multi_t \
        --out_json /kaggle/working/eval_multi_t.json
!ls -la {OUT}


Al terminar: **Save Version → Quick Save (Save output)**.  En la siguiente sesión añade esta versión como *Input* (Your Work → Notebooks), pon `RESUME = glob.glob('/kaggle/input/*/checkpoints_run2/last.pth')[0]` y deja `INIT_FROM = None` (con `--resume` se ignora).

Resultados esperados: la validación t=0.5 del run 2 **no** debería bajar del run 1 (arranca idéntico); el paper reporta RIFE-m ~0.1 dB por debajo de RIFE en t=0.5 a cambio de t arbitrario, y el EMA + scale-aug deberían compensarlo.  Lo que sí cambia radicalmente es la evaluación `--multi_t`: un RIFE clásico se hunde en t=1/6 y 5/6; RIFE-m debe dar PSNR parecido en todos los t.

Después, prueba el modelo con vídeos reales: `python demo_video.py --ckpt checkpoints_run2/inference.pth --source <youtube|mp4> --multi 3` (ver `docs/demo_sources.txt`).
